In [8]:
file_path = r"C:\Users\Brandon\Documents\repo\doc-processor\rnd\doc\statement.pdf"

# Approach 1 : Veryfi Webservice

In [2]:
!pip install veryfi


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from veryfi import Client
from typing import Dict, Any

VERYFI_CLIENT_ID="vrfRuvSDf8Z8Lnf8UAKEsAtiBrvyXsJHa1SL0ji"
VERYFI_CLIENT_SECRET="5FI7NxrNBzMS5nyTZB8NOX15wCtR1oE6BOeOtfVPO3VNRs7xYk5wM1MyONEDdfL0ed90OgXyC0PxnPLGk3Ou2dOIfeiHrU2RUn5ujBLc2FBisNzaNfvzf8VLYbfzPb9v"
VERYFI_USERNAME="jinsiang.work"
VERYFI_API_KEY="f4bfcbe874e59145a5793c729e5e47a3"

client = Client(
    client_id=VERYFI_CLIENT_ID,
    client_secret=VERYFI_CLIENT_SECRET,
    username=VERYFI_USERNAME,
    api_key=VERYFI_API_KEY
)

In [16]:
def process_document_veryfi(file_path: str) -> Dict[str, Any]:
    result = client.process_document(
        file_path,
        categories=["bank_statement"]
    )

    print(f"Veryfi processing complete: {result}")
    return result

def extract_bank_statement_data_veryfi(veryfi_result: Dict[str, Any]) -> Dict[str, Any]:
    vendor = veryfi_result.get("vendor", {})
    bill_to = veryfi_result.get("bill_to", {})
    meta = veryfi_result.get("meta", {})

    extracted_data = {
        "bank_name": vendor.get("name") if isinstance(vendor, dict) else None,
        "bank_address": vendor.get("address") if isinstance(vendor, dict) else None,
        "account_holder_name": bill_to.get("name") if isinstance(bill_to, dict) else None,
        "account_holder_address": bill_to.get("address") if isinstance(bill_to, dict) else None,
        "account_number": veryfi_result.get("invoice_number"),  # Account number stored as invoice_number
        "statement_date": veryfi_result.get("date"),
        "currency": veryfi_result.get("currency_code"),
        "beginning_balance": None,
        "ending_balance": None,
        "total_transactions": len(veryfi_result.get("line_items", [])),
        "ocr_confidence": meta.get("ocr_score", 0.0) if isinstance(meta, dict) else 0.0,
        "document_type": veryfi_result.get("document_type"),
        "reference_number": veryfi_result.get("reference_number"),
    }

    return extracted_data

def display_veryfi_result(result: Dict[str, Any]):
    import json
    print(json.dumps(result, indent=2))

In [17]:
result = process_document_veryfi(file_path)
print("Raw Veryfi Result:")
print(result)
print("\n" + "="*50 + "\n")

# Extract bank statement data
extracted = extract_bank_statement_data_veryfi(result)
print("Extracted Bank Statement Data:")
for key, value in extracted.items():
    print(f"{key}: {value}")

Veryfi processing complete: {'account_number': None, 'bill_to': {'address': 'PERSIARAN KEWAJIPAN USJ 19\n47620 SUBANG JAYA', 'name': 'SHARINAZ NORLINA BINTI AHMAD AL BAKISH', 'parsed_address': None, 'vat_number': None}, 'cashback': None, 'category': None, 'country_code': 'MY', 'created_date': '2025-10-23 02:18:30', 'currency_code': 'MYR', 'date': '2021-12-31 00:00:00', 'delivery_date': None, 'discount': None, 'document_reference_number': None, 'document_title': None, 'document_type': 'receipt', 'due_date': None, 'duplicate_of': 364942308, 'external_id': None, 'id': 365298718, 'img_file_name': '365298718.pdf', 'img_thumbnail_url': 'https://scdn.veryfi.com/receipts/996652c180117871/d0b6b952-ae7f-43c7-a8be-e36ae39c5411/thumbnail.jpg?Expires=1761186813&Signature=H3MkgwJa656YefzT8Gxaw7YJ6K-RS8A~WdXEsyJWKVbcv~w6~2Tavrea~kosOogJqjsmylGMJIrMLkcz5oRXslrsO1kFISZcgDPqa2ZyfiooNAF1WIb4fX0aYCd4PQOlKjg4TZ0HPzsW2ZsOglDpPO6u5hP6SgjjODFsZRsLR54Xvz0MLn4grh8EXQHqsdSx8ShwWQ40gj8stwzGcU-aN1y9I5el4v410oWs448

# Approach 2: FinHero Webservice

In [18]:
!pip install requests python-dotenv

  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()  # load your API key from .env

API_KEY = ""
BASE_URL = "https://finapi.developer.azure-api.net"
TIMEOUT = 60  # seconds

def process_document_finhero(file_path: str):
    """
    Uploads a bank statement PDF/image to FinXtract API
    and returns JSON response.
    """
    if not API_KEY:
        raise ValueError("Missing FINXTRACT_API_KEY")

    url = f"{BASE_URL}/finxtract/bankstatement/v2/extract"
    headers = {
        "Ocp-Apim-Subscription-Key": API_KEY,
        "Accept": "application/json"
    }

    with open(file_path, "rb") as f:
        files = {"file": (os.path.basename(file_path), f, "application/pdf")}
        response = requests.post(url, headers=headers, files=files, timeout=TIMEOUT)

    response.raise_for_status()
    return response.json()

def extract_bank_statement_data_finhero(finhero_result) -> Dict[str, Any]:
    vendor = finhero_result.get("vendor", {})
    bill_to = finhero_result.get("bill_to", {})
    meta = finhero_result.get("meta", {})

    extracted_data = {
        "bank_name": vendor.get("name") if isinstance(vendor, dict) else None,
        "bank_address": vendor.get("address") if isinstance(vendor, dict) else None,
        "account_holder_name": bill_to.get("name") if isinstance(bill_to, dict) else None,
        "account_holder_address": bill_to.get("address") if isinstance(bill_to, dict) else None,
        "account_number": finhero_result.get("invoice_number"),  # Account number stored as invoice_number
        "statement_date": finhero_result.get("date"),
        "currency": finhero_result.get("currency_code"),
        "beginning_balance": None,
        "ending_balance": None,
        "total_transactions": len(finhero_result.get("line_items", [])),
        "ocr_confidence": meta.get("ocr_score", 0.0) if isinstance(meta, dict) else 0.0,
        "document_type": finhero_result.get("document_type"),
        "reference_number": finhero_result.get("reference_number"),
    }

    return extracted_data

In [ ]:
result = process_document_finhero(file_path)
print("Raw FinHero Result:")
print(result)
print("\n" + "="*50 + "\n")

# Extract bank statement data
extracted = extract_bank_statement_data_finhero(result)
print("Extracted Bank Statement Data:")
for key, value in extracted.items():
    print(f"{key}: {value}")

# Approach 3: